# SQL Window Functions — Complete Reference

| Pattern | Function |
|---------|----------|
| Ranking | ROW_NUMBER / RANK / DENSE_RANK |
| Lag/Lead | LAG / LEAD for time-series |
| Running totals | SUM / AVG OVER (ORDER BY) |
| Percentiles | NTILE / PERCENT_RANK |
| Frame spec | ROWS vs RANGE BETWEEN |

**Mental model**: Window functions compute a value for each row using a *set of related rows* (the window) — without collapsing rows like GROUP BY does.

```
OVER (
  PARTITION BY col   -- define peer groups
  ORDER BY col       -- define ordering within group
  ROWS/RANGE BETWEEN -- define frame boundaries
)
```

## Visual Model

```
RAW TABLE: sales(employee, region, month, amount)

  emp   region  month  amount
  ----  ------  -----  ------
  Alice  East    1      100       ← PARTITION by region = 'East'
  Alice  East    2      200       ←
  Alice  East    3      150       ←
  Bob    West    1      300       ← PARTITION by region = 'West'
  Bob    West    2      250       ←

WINDOW FUNCTION: SUM(amount) OVER (PARTITION BY region ORDER BY month)

  emp   region  month  amount  running_sum
  ----  ------  -----  ------  -----------
  Alice  East    1      100     100        ← frame: rows 1..current
  Alice  East    2      200     300        ← frame: rows 1..current
  Alice  East    3      150     450        ← frame: rows 1..current
  Bob    West    1      300     300        ← new partition, reset
  Bob    West    2      250     550        ←

KEY: rows are PRESERVED (unlike GROUP BY); window value added per row

RANKING FUNCTIONS (no frame needed):
  ROW_NUMBER: 1,2,3,4,...  always unique
  RANK:       1,1,3,...    ties share rank, gap after
  DENSE_RANK: 1,1,2,...    ties share rank, no gap
```

## Setup — Libraries and Config

In [ ]:
import sqlite3
import textwrap

# SQLite 3.25+ supports all standard window functions
print(f"sqlite3 version: {sqlite3.sqlite_version}")

def make_db():
    """Create fresh in-memory database with sample data."""
    conn = sqlite3.connect(":memory:")
    conn.row_factory = sqlite3.Row
    cur = conn.cursor()

    cur.executescript("""
        CREATE TABLE sales (
            sale_id   INTEGER PRIMARY KEY,
            employee  TEXT,
            region    TEXT,
            month     INTEGER,
            amount    REAL
        );
        INSERT INTO sales VALUES
            (1,'Alice','East',1,100),(2,'Alice','East',2,200),
            (3,'Alice','East',3,150),(4,'Alice','East',4,200),
            (5,'Bob','East',1,300),(6,'Bob','East',2,250),
            (7,'Bob','East',3,300),(8,'Bob','East',4,100),
            (9,'Carol','West',1,500),(10,'Carol','West',2,400),
            (11,'Carol','West',3,600),(12,'Carol','West',4,450),
            (13,'Dave','West',1,200),(14,'Dave','West',2,350),
            (15,'Dave','West',3,300),(16,'Dave','West',4,400);

        CREATE TABLE stock_prices (
            ticker TEXT,
            trade_date TEXT,
            close_price REAL
        );
        INSERT INTO stock_prices VALUES
            ('AAPL','2024-01-02',185.2),('AAPL','2024-01-03',184.1),
            ('AAPL','2024-01-04',181.9),('AAPL','2024-01-05',186.0),
            ('AAPL','2024-01-08',188.3),
            ('MSFT','2024-01-02',374.0),('MSFT','2024-01-03',375.5),
            ('MSFT','2024-01-04',370.2),('MSFT','2024-01-05',378.1),
            ('MSFT','2024-01-08',380.0);
    """)
    conn.commit()
    return conn

def run(conn, sql, title=""):
    """Execute SQL and print results as a table."""
    cur = conn.execute(sql)
    rows = cur.fetchall()
    if not rows:
        print(f"{title}: (no rows)")
        return []
    cols = [d[0] for d in cur.description]
    col_w = [max(len(c), max(len(str(r[c])) for r in rows)) for c in cols]
    sep = "  ".join("-" * w for w in col_w)
    header = "  ".join(c.ljust(w) for c, w in zip(cols, col_w))
    if title:
        print(f"\n=== {title} ===")
    print(header)
    print(sep)
    for r in rows:
        print("  ".join(str(r[c]).ljust(w) for c, w in zip(cols, col_w)))
    return rows

conn = make_db()
print("Database ready.")

## Decision Map — Which Window Function?

```
What do you need?
│
├─ Rank rows within a group?
│   ├─ Need unique rank (no ties)?    → ROW_NUMBER
│   ├─ Ties share rank, gap after?    → RANK
│   └─ Ties share rank, no gap?       → DENSE_RANK
│
├─ Compare row to previous/next row?
│   ├─ Access prior row value?        → LAG(col, offset, default)
│   └─ Access next row value?         → LEAD(col, offset, default)
│
├─ Cumulative / rolling aggregate?
│   ├─ Running total?                 → SUM OVER (ORDER BY)
│   ├─ Rolling N-row average?         → AVG OVER (ROWS BETWEEN N PRECEDING AND CURRENT ROW)
│   └─ Entire partition aggregate?    → SUM OVER (PARTITION BY) [no ORDER BY]
│
├─ Percentile / bucket assignment?
│   ├─ Split into N equal buckets?    → NTILE(N)
│   └─ 0.0–1.0 relative position?    → PERCENT_RANK / CUME_DIST
│
└─ Frame boundary matters?
    ├─ Physical row count?            → ROWS BETWEEN
    └─ Value range?                   → RANGE BETWEEN (needs ORDER BY numeric/date)
```

## Pattern 1 — ROW_NUMBER / RANK / DENSE_RANK

In [ ]:
# Ranking employees within each region by total sales
# Shows difference between ROW_NUMBER, RANK, DENSE_RANK when ties exist

conn2 = make_db()

# Create a view with tied totals to demonstrate ranking differences
conn2.execute("""
    CREATE VIEW emp_totals AS
    SELECT employee, region, SUM(amount) AS total_sales
    FROM sales
    GROUP BY employee, region
""")

# Force a tie: set Bob East total = Alice East total (both = 650)
conn2.execute("INSERT INTO sales VALUES (17,'Bob2','East',1,250),(18,'Bob2','East',2,200),(19,'Bob2','East',3,100),(20,'Bob2','East',4,100)")
conn2.execute("""
    CREATE VIEW emp_totals2 AS
    SELECT employee, region, SUM(amount) AS total_sales
    FROM sales
    GROUP BY employee, region
""")

run(conn2, """
    SELECT
        employee,
        region,
        total_sales,
        ROW_NUMBER() OVER (PARTITION BY region ORDER BY total_sales DESC) AS row_num,
        RANK()       OVER (PARTITION BY region ORDER BY total_sales DESC) AS rnk,
        DENSE_RANK() OVER (PARTITION BY region ORDER BY total_sales DESC) AS dense_rnk
    FROM emp_totals2
    ORDER BY region, total_sales DESC
""", "Ranking with ties")

# Practical use: Top-1 per region (no duplicates if tied — ROW_NUMBER is deterministic)
run(conn, """
    SELECT employee, region, total_sales
    FROM (
        SELECT
            employee, region,
            SUM(amount) AS total_sales,
            ROW_NUMBER() OVER (PARTITION BY region ORDER BY SUM(amount) DESC) AS rn
        FROM sales
        GROUP BY employee, region
    )
    WHERE rn = 1
""", "Top seller per region")

## Pattern 2 — LAG / LEAD (Time-Series Analysis)

In [ ]:
# LAG: access previous row's value within same partition
# LEAD: access next row's value within same partition
# Usage: period-over-period change, day-over-day return

run(conn, """
    SELECT
        ticker,
        trade_date,
        close_price,
        LAG(close_price, 1, NULL) OVER (
            PARTITION BY ticker
            ORDER BY trade_date
        ) AS prev_close,
        ROUND(
            100.0 * (close_price - LAG(close_price,1,close_price) OVER (
                PARTITION BY ticker ORDER BY trade_date
            )) / LAG(close_price,1,close_price) OVER (
                PARTITION BY ticker ORDER BY trade_date
            ), 2
        ) AS pct_change,
        LEAD(close_price, 1, NULL) OVER (
            PARTITION BY ticker
            ORDER BY trade_date
        ) AS next_close
    FROM stock_prices
    ORDER BY ticker, trade_date
""", "Daily price change with LAG/LEAD")

# Month-over-month sales change per employee
run(conn, """
    SELECT
        employee,
        month,
        amount,
        LAG(amount) OVER (PARTITION BY employee ORDER BY month) AS prev_month,
        amount - LAG(amount, 1, 0) OVER (PARTITION BY employee ORDER BY month) AS delta
    FROM sales
    ORDER BY employee, month
""", "Month-over-month delta per employee")

## Pattern 3 — Running SUM / AVG (Cumulative Aggregates)

In [ ]:
# Running totals: SUM OVER (ORDER BY) — frame defaults to RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
# Rolling average: explicit ROWS BETWEEN to use physical row count

run(conn, """
    SELECT
        employee,
        month,
        amount,
        SUM(amount) OVER (
            PARTITION BY employee
            ORDER BY month
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS running_total,
        ROUND(AVG(amount) OVER (
            PARTITION BY employee
            ORDER BY month
            ROWS BETWEEN 1 PRECEDING AND 1 FOLLOWING
        ), 1) AS rolling_3m_avg,
        ROUND(100.0 * SUM(amount) OVER (
            PARTITION BY employee
            ORDER BY month
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) / SUM(amount) OVER (PARTITION BY employee), 1) AS pct_of_yearly
    FROM sales
    WHERE employee = 'Alice'
    ORDER BY employee, month
""", "Running totals and rolling averages")

# Total per region without collapsing rows (vs GROUP BY)
run(conn, """
    SELECT
        employee,
        region,
        month,
        amount,
        SUM(amount) OVER (PARTITION BY region) AS region_total,
        ROUND(100.0 * amount / SUM(amount) OVER (PARTITION BY region), 1) AS pct_of_region
    FROM sales
    ORDER BY region, employee, month
    LIMIT 8
""", "Row-level pct of region total (no GROUP BY)")

## Pattern 4 — NTILE / PERCENT_RANK

In [ ]:
# NTILE(N): divides rows into N equal-sized buckets (1=top)
# PERCENT_RANK: (rank - 1) / (total rows - 1) → 0.0 to 1.0
# CUME_DIST: cumulative distribution → fraction of rows <= current

run(conn, """
    SELECT
        employee,
        region,
        total_sales,
        NTILE(4) OVER (ORDER BY total_sales DESC) AS quartile,
        ROUND(PERCENT_RANK() OVER (ORDER BY total_sales), 3) AS pct_rank,
        ROUND(CUME_DIST() OVER (ORDER BY total_sales), 3) AS cume_dist
    FROM (
        SELECT employee, region, SUM(amount) AS total_sales
        FROM sales
        GROUP BY employee, region
    )
    ORDER BY total_sales DESC
""", "NTILE / PERCENT_RANK / CUME_DIST")

# Practical: label performance tiers
run(conn, """
    SELECT
        employee,
        total_sales,
        CASE NTILE(3) OVER (ORDER BY total_sales DESC)
            WHEN 1 THEN 'Top'
            WHEN 2 THEN 'Mid'
            WHEN 3 THEN 'Low'
        END AS tier
    FROM (
        SELECT employee, SUM(amount) AS total_sales
        FROM sales
        GROUP BY employee
    )
    ORDER BY total_sales DESC
""", "Performance tier assignment")

## Pattern 5 — Frame Specification (ROWS vs RANGE BETWEEN)

In [ ]:
# ROWS BETWEEN: physical row boundaries — always predictable
# RANGE BETWEEN: value boundaries — includes all rows with same ORDER BY value
# Key difference: RANGE with ties can include unexpected rows

conn3 = sqlite3.connect(":memory:")
conn3.row_factory = sqlite3.Row
conn3.executescript("""
    CREATE TABLE scores(student TEXT, score INTEGER);
    INSERT INTO scores VALUES
        ('A',50),('B',50),('C',70),('D',70),('E',90);
""")

run(conn3, """
    SELECT
        student,
        score,
        SUM(score) OVER (
            ORDER BY score
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS rows_running,
        SUM(score) OVER (
            ORDER BY score
            RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS range_running
    FROM scores
""", "ROWS vs RANGE with ties (score=50 appears twice)")

print("""
Explanation:
  ROWS: frame ends at CURRENT ROW's physical position — B gets only A+B
  RANGE: frame includes ALL rows with score <= current score
         — both A and B get A+B (same score=50 → same RANGE boundary)

Frame shorthand:
  UNBOUNDED PRECEDING → start of partition
  N PRECEDING         → N rows/values before current
  CURRENT ROW         → current row (default end for running totals)
  N FOLLOWING         → N rows/values after current
  UNBOUNDED FOLLOWING → end of partition
""")

# Common frames:
run(conn, """
    SELECT
        employee, month, amount,
        -- 3-month centered rolling average
        ROUND(AVG(amount) OVER (
            PARTITION BY employee ORDER BY month
            ROWS BETWEEN 1 PRECEDING AND 1 FOLLOWING
        ), 1) AS centered_3m,
        -- Last 2 months trailing average
        ROUND(AVG(amount) OVER (
            PARTITION BY employee ORDER BY month
            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
        ), 1) AS trailing_3m,
        -- All-time to date
        SUM(amount) OVER (
            PARTITION BY employee ORDER BY month
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS ytd
    FROM sales
    WHERE employee = 'Carol'
    ORDER BY month
""", "Common frame patterns")

## Full Decision Map

```
WINDOW FUNCTION SELECTION
─────────────────────────
Goal: rank rows           → ROW_NUMBER (unique) / RANK (gaps) / DENSE_RANK (no gaps)
Goal: prior/next value    → LAG(col, n, default) / LEAD(col, n, default)
Goal: cumulative sum      → SUM() OVER (ORDER BY ... ROWS UNBOUNDED PRECEDING)
Goal: rolling N-row avg   → AVG() OVER (ROWS BETWEEN N PRECEDING AND CURRENT ROW)
Goal: partition total     → SUM() OVER (PARTITION BY) [no ORDER BY → entire partition]
Goal: bucket assignment   → NTILE(n) OVER (ORDER BY ...)
Goal: relative position   → PERCENT_RANK() or CUME_DIST()
Goal: first/last in group → FIRST_VALUE() / LAST_VALUE() OVER (...)

FRAME RULES
───────────
ROWS BETWEEN   → physical rows (use this for rolling windows — predictable)
RANGE BETWEEN  → value range (default; affected by ties — use carefully)
No ORDER BY    → entire partition is the frame

PERFORMANCE TIPS
────────────────
1. Multiple OVER() clauses on same PARTITION+ORDER = one sort, reused
2. Avoid DISTINCT inside window subquery — use CTE to pre-aggregate first
3. Filter AFTER window (in outer WHERE) — window sees full partition
4. For top-N per group: ROW_NUMBER in CTE → WHERE rn <= N in outer query
5. RANGE BETWEEN on non-integer columns can be slow (type casting)
```

## Cheat Sheet

```sql
-- Ranking
ROW_NUMBER() OVER (PARTITION BY dept ORDER BY salary DESC)
RANK()       OVER (PARTITION BY dept ORDER BY salary DESC)  -- gaps on tie
DENSE_RANK() OVER (PARTITION BY dept ORDER BY salary DESC)  -- no gaps

-- Lag / Lead
LAG(col, 1, 0)  OVER (PARTITION BY grp ORDER BY dt)  -- prev row, default 0
LEAD(col, 1, 0) OVER (PARTITION BY grp ORDER BY dt)  -- next row, default 0

-- Running / Rolling
SUM(col) OVER (PARTITION BY grp ORDER BY dt
    ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)       -- YTD
AVG(col) OVER (PARTITION BY grp ORDER BY dt
    ROWS BETWEEN 6 PRECEDING AND CURRENT ROW)               -- 7-day rolling
SUM(col) OVER (PARTITION BY grp)                            -- group total, all rows

-- Percentile / Bucket
NTILE(4)       OVER (ORDER BY col)   -- quartile 1..4
PERCENT_RANK() OVER (ORDER BY col)   -- 0.0 .. 1.0
CUME_DIST()    OVER (ORDER BY col)   -- fraction <= current

-- First/Last value in window
FIRST_VALUE(col) OVER (PARTITION BY grp ORDER BY dt)
LAST_VALUE(col)  OVER (PARTITION BY grp ORDER BY dt
    ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING)

-- Top-N per group pattern
SELECT * FROM (
    SELECT *, ROW_NUMBER() OVER (PARTITION BY dept ORDER BY salary DESC) rn
    FROM employees
) WHERE rn <= 3;

-- Period-over-period change
SELECT month, revenue,
    revenue - LAG(revenue) OVER (ORDER BY month) AS delta,
    ROUND(100.0*(revenue - LAG(revenue) OVER (ORDER BY month))
        / LAG(revenue) OVER (ORDER BY month), 1) AS pct_chg
FROM monthly_revenue;
```

## Summary Map

```
WINDOW FUNCTIONS — ONE-PAGE SUMMARY
────────────────────────────────────

Anatomy:  func() OVER (PARTITION BY ... ORDER BY ... frame_clause)

RANKING ──────────────────────────────────────────────────────────
  ROW_NUMBER  always unique, arbitrary tiebreak
  RANK        1,1,3 — gap after tie
  DENSE_RANK  1,1,2 — no gap after tie
  Use ROW_NUMBER for top-N per group (guarantees exactly N rows)

OFFSET ───────────────────────────────────────────────────────────
  LAG(col,n,default)   previous nth row in partition
  LEAD(col,n,default)  next nth row in partition
  Default arg avoids NULL on first/last row

AGGREGATE OVER WINDOW ────────────────────────────────────────────
  No ORDER BY → entire partition (like a join to GROUP BY result)
  ORDER BY + default frame → RANGE UNBOUNDED PRECEDING (running total)
  Use ROWS BETWEEN for rolling windows (avoid RANGE tie surprises)

DISTRIBUTION ─────────────────────────────────────────────────────
  NTILE(n)      bucket 1..n
  PERCENT_RANK  (rank-1)/(rows-1) → 0..1
  CUME_DIST     fraction of rows <= current → 0..1

INTERVIEW SIGNALS ────────────────────────────────────────────────
  ✓ Know RANK vs DENSE_RANK tie behavior
  ✓ Know ROWS vs RANGE difference
  ✓ Top-N per group = ROW_NUMBER in CTE + WHERE rn <= N
  ✓ Period-over-period = LAG in CTE or subquery
  ✓ Running total = SUM OVER (ORDER BY ROWS UNBOUNDED PRECEDING)
```